# 01 — Data Inspection & Cleaning

**Project:** Effects of the North Frontier Economic Free Zone (NFZ) on IMMEX employment — Synthetic Control Method

**Author:** Hector Alejandro Vazquez Reyes

This notebook prepares the panel dataset used in the SCM analysis. It performs four tasks:

1. Load the raw panel (41 municipalities × monthly observations, July 2007 – January 2022)
2. Rename variables to readable names and construct derived variables (`log_immex_emp`, `som`, `sop`)
3. Run basic data-quality checks (duplicates, panel balance, internal consistency)
4. Save the clean panel to parquet for use in subsequent notebooks

**Policy context.** In January 2019, the Mexican federal government created the North Frontier
Economic Free Zone: a VAT reduction (16% → 8%), an income-tax reduction (30% → 20%), and a
100% minimum-wage increase applied to 43 municipalities along the US border. This analysis
estimates the effect of that policy package on employment in the IMMEX
(export-manufacturing) sector using the Synthetic Control Method.

In [1]:
import os
import pandas as pd
import numpy as np

os.chdir("/Users/alexvr/GITHUB/nfz_scm_app")

df = pd.read_excel("data/raw/nfz_panel_2007_2022.xlsx")
df.shape

(7175, 38)

## 1. Rename variables

The raw file uses the short variable names from the original Stata analysis (e.g. `emp`, `pea`, `v25`).
We rename them to self-explanatory names so the code reads without a codebook.

| Raw name | New name | Description |
|---|---|---|
| `est` | `establishments` | Number of IMMEX establishments |
| `emp` | `immex_emp` | Total IMMEX employment *(main outcome)* |
| `mf` | `manuf_emp` | Manufacturing-sector employment |
| `pea` | `econ_active_pop` | Economically active population |
| `twh` | `total_work_hours` | Total work hours |
| `tp` | `personnel_payments` | Total payments to personnel |
| `wdm` | `worked_days_manuf` | Worked days of manufacturer |
| `ni` | `income_national` | Income from national market |
| `fi` | `income_international` | Income from international market |
| `pt` | `prod_services_payments` | Payments for production-related services |
| `v25` | `consumed_goods_services_nm` | Consumed goods/services, national market |

In [2]:
df.columns = df.columns.str.strip().str.lower()

df = df.rename(columns={
    "est": "establishments",
    "emp": "immex_emp",
    "mf": "manuf_emp",
    "pea": "econ_active_pop",
    "twh": "total_work_hours",
    "tp": "personnel_payments",
    "wdm": "worked_days_manuf",
    "ni": "income_national",
    "fi": "income_international",
    "pt": "prod_services_payments",
    "v25": "consumed_goods_services_nm"
})

# itaee (state-level quarterly economic activity index) is 100% missing in this file — drop it
df = df.drop(columns=["itaee"], errors="ignore")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7175 entries, 0 to 7174
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   yr                          7175 non-null   int64  
 1   m                           7175 non-null   int64  
 2   date                        7175 non-null   object 
 3   mun                         7175 non-null   object 
 4   establishments              7175 non-null   int64  
 5   est_201912                  7175 non-null   int64  
 6   lp                          7175 non-null   float64
 7   ap                          7175 non-null   float64
 8   lpo                         7175 non-null   int64  
 9   apo                         7175 non-null   int64  
 10  immex_emp                   7175 non-null   int64  
 11  manuf_emp                   7175 non-null   int64  
 12  econ_active_pop             7175 non-null   int64  
 13  twh_l                       7175 

## 2. Build the time index and sort the panel

The raw file stores year (`yr`) and month (`m`) as separate integers. We build a proper
datetime column and sort by municipality and date so the panel has a consistent structure.

In [3]:
df["date"] = pd.to_datetime(dict(year=df["yr"].astype(int), month=df["m"].astype(int), day=1))
df["mun"] = df["mun"].str.strip()

df = df.sort_values(["mun", "date"]).reset_index(drop=True)
df[["mun", "date"]].head()

,mun,date
0,Acuna,2007-07-01
1,Acuna,2007-08-01
2,Acuna,2007-09-01
3,Acuna,2007-10-01
4,Acuna,2007-11-01


## 3. Derived variables

Following the thesis specification:

- `log_immex_emp` — log of IMMEX employment, the main outcome in the log-employment model
- `log_eap` — log of the economically active population
- `som` — share of IMMEX employment over **manufacturing** employment: `(immex_emp / manuf_emp) × 100`
- `sop` — share of IMMEX employment over the **total economically active population**: `(immex_emp / econ_active_pop) × 100`. This is the key outcome in the SOP model.

In [4]:
df["log_immex_emp"] = np.log(df["immex_emp"])
df["log_eap"] = np.log(df["econ_active_pop"])

df["som"] = (df["immex_emp"] / df["manuf_emp"]) * 100
df["sop"] = (df["immex_emp"] / df["econ_active_pop"]) * 100

df[["immex_emp", "manuf_emp", "econ_active_pop", "som", "sop"]].describe()

,immex_emp,manuf_emp,econ_active_pop,som,sop
count,7175.000000,7175.000000,7.175000e+03,7175.000000,7175.000000
mean,48239.181324,82295.157770,4.640630e+05,63.742901,13.959540
std,46098.394622,62953.009092,3.113011e+05,24.427618,13.362438
min,5242.000000,10409.000000,3.508600e+04,8.723339,0.850055
25%,21683.500000,36047.000000,2.379260e+05,49.272230,5.688536
50%,32143.000000,63733.000000,3.938690e+05,71.904261,10.571473
75%,57046.000000,108777.500000,6.029795e+05,82.497088,17.862361
max,303573.000000,403919.000000,1.866767e+06,96.662849,141.281293


## 4. Data-quality checks

Three checks before we trust the panel:

1. **Internal consistency** — IMMEX employment should not exceed manufacturing employment,
   and manufacturing employment should not exceed the economically active population.
2. **No duplicate municipality-month rows.**
3. **Panel balance** — every municipality should have the same number of monthly observations.

In [5]:
print("Any IMMEX > Manufacturing?",
      (df["immex_emp"] > df["manuf_emp"]).any())

print("Any Manufacturing > EAP?",
      (df["manuf_emp"] > df["econ_active_pop"]).any())

Any IMMEX > Manufacturing? False
Any Manufacturing > EAP? True


In [6]:
# Inspect the violations of manuf_emp <= econ_active_pop, if any.
# These arise because the two series come from different sources (IMMEX program data
# vs ENOE survey) with different coverage and timing.
viol = df[df["manuf_emp"] > df["econ_active_pop"]].copy()
viol["ratio"] = viol["manuf_emp"] / viol["econ_active_pop"]

print("Number of violations:", len(viol))
if len(viol) > 0:
    print("Max ratio:", viol["ratio"].max())
    print("Median ratio:", viol["ratio"].median())
    display(viol[["mun", "date", "manuf_emp", "econ_active_pop"]].head(10))

Number of violations: 28
Max ratio: 1.7090509642246874
Median ratio: 1.0527111291737195


,mun,date,manuf_emp,econ_active_pop
5155,Ramos Arizpe,2014-03-01,53581,52404
5156,Ramos Arizpe,2014-04-01,54310,52718
5157,Ramos Arizpe,2014-05-01,54209,53032
5163,Ramos Arizpe,2014-11-01,54546,54362
5164,Ramos Arizpe,2014-12-01,54303,54267
5165,Ramos Arizpe,2015-01-01,54186,54153
5167,Ramos Arizpe,2015-03-01,54490,53926
5168,Ramos Arizpe,2015-04-01,54855,54622
5190,Ramos Arizpe,2017-02-01,59967,58388
5191,Ramos Arizpe,2017-03-01,62333,57181


In [7]:
dups = df.duplicated(subset=["mun", "date"]).sum()
print("Duplicated mun-date rows:", dups)

counts = df.groupby("mun")["date"].nunique()
print("Municipalities:", df["mun"].nunique())
print("Date range:", df["date"].min().date(), "→", df["date"].max().date())
print("Months per municipality (min / median / max):",
      counts.min(), "/", counts.median(), "/", counts.max())

Duplicated mun-date rows: 0
Municipalities: 41
Date range: 2007-07-01 → 2022-01-01
Months per municipality (min / median / max): 175 / 175.0 / 175


## 5. Treatment assignment

The NFZ policy took effect in **January 2019**. The 10 treated municipalities in this study
are the IMMEX-relevant municipalities inside the free zone:

Acuña, Ensenada, Ciudad Juárez, Matamoros, Mexicali, Nogales, Nuevo Laredo, Reynosa, Tecate, Tijuana.

All remaining municipalities form the donor pool for the synthetic controls.

We create three flags:
- `treated` — 1 if the municipality is in the NFZ
- `post` — 1 from January 2019 onward
- `treated_post` — interaction of the two

In [8]:
treated_muns = [
    "Acuna", "Ensenada", "Juarez", "Matamoros", "Mexicali",
    "Nogales", "Nuevo Laredo", "Reynosa", "Tecate", "Tijuana"
]

treatment_date = pd.Timestamp("2019-01-01")

df["treated"] = df["mun"].isin(treated_muns).astype(int)
df["post"] = (df["date"] >= treatment_date).astype(int)
df["treated_post"] = df["treated"] * df["post"]

df.groupby("treated")["mun"].nunique().rename("n_municipalities")

treated
0    31
1    10
Name: n_municipalities, dtype: int64

## 6. Save the clean panel

The processed panel is saved to parquet and re-read as a verification step.
All subsequent notebooks load this file.

In [9]:
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/processed/nfz_panel_clean.parquet", index=False)
print("Saved nfz_panel_clean.parquet successfully")

Saved nfz_panel_clean.parquet successfully


In [10]:
# Verification: re-read and confirm key columns survived the round trip
df_check = pd.read_parquet("data/processed/nfz_panel_clean.parquet")

assert df_check.shape == df.shape, "Shape mismatch after round trip"
assert {"som", "sop", "log_immex_emp", "treated", "post"}.issubset(df_check.columns)

df_check[["mun", "date", "immex_emp", "som", "sop", "treated", "post"]].head()

,mun,date,immex_emp,som,sop,treated,post
0,Acuna,2007-07-01,24934,67.568153,33.210352,1,0
1,Acuna,2007-08-01,25706,69.109582,34.409552,1,0
2,Acuna,2007-09-01,25924,69.540492,34.875024,1,0
3,Acuna,2007-10-01,26360,70.396582,35.639923,1,0
4,Acuna,2007-11-01,25556,68.811761,34.727545,1,0


---
## Summary

- Panel: **41 municipalities × monthly observations, July 2007 – January 2022**, balanced, no duplicates
- 10 treated municipalities (NFZ), 31 donor-pool municipalities
- Variables renamed to readable names; `som`, `sop`, and log transforms constructed per the thesis specification
- Clean panel saved to `data/processed/nfz_panel_clean.parquet`

**Next:** `02_scm_core.ipynb` — construction of the predictor matrices and the synthetic control optimizer.